# Domain C: Contextual Adaptation and History Dependence

This notebook addresses three questions:
- **C1:** Do responses differ between the first and second presentation of the same block type (context-dependent adaptation)?
- **C2:** Does the context (contrast-varying vs. speed-varying) alter tuning curves?
- **C3:** Do responses change across days (multi-day representational drift)?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).
Context structure: Blocks 0,2 = contrast-context (TF fixed at 1 Hz); Blocks 1,3 = speed-context (contrast fixed at 0.8).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr, ttest_rel
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import get_block_responses, compute_adaptation_index, get_trial_position_responses, exp_decay, linear_CKA
from functions.tuning import naka_rushton, normalization_model

sns.set_context('talk')
sns.set_style('whitegrid')

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

print(f"Total cells: {len(obs)}, Total trials: {X.shape[1]}")

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

## C1: Block-to-Block Context-Dependent Adaptation

Compare responses to identical stimuli between Block 0 vs Block 2 (contrast context) and Block 1 vs Block 3 (speed context). Compute adaptation indices. Test cell-type specificity.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.1  Compute mean response per cell per stimulus condition in each block
# ══════════════════════════════════════════════════════════════════════

# Contrast-context blocks (0 vs 2): contrast varies, TF = 1 Hz
resp_block0 = get_block_responses(X, var, 0.0, 'contrast', contrasts)
resp_block2 = get_block_responses(X, var, 2.0, 'contrast', contrasts)

# Speed-context blocks (1 vs 3): TF varies, contrast = 0.8
resp_block1 = get_block_responses(X, var, 1.0, 'temporal_frequency', tfs)
resp_block3 = get_block_responses(X, var, 3.0, 'temporal_frequency', tfs)

# ── Collapse over all conditions to get a single adaptation index per cell ──
contrast_conditions = [(ori, c) for ori in orientations for c in contrasts]
speed_conditions = [(ori, tf) for ori in orientations for tf in tfs]

adapt_idx_contrast = compute_adaptation_index(resp_block0, resp_block2, contrast_conditions)
adapt_idx_speed = compute_adaptation_index(resp_block1, resp_block3, speed_conditions)

adapt_df = pd.DataFrame({
    'subclass': obs['subclass_name'].values,
    'subclass_short': obs['subclass_name'].map(SUBCLASS_SHORT).values,
    'mouse_id': obs['mouse_id'].values,
    'adapt_contrast': adapt_idx_contrast,
    'adapt_speed': adapt_idx_speed,
})
adapt_df['adapt_mean'] = adapt_df[['adapt_contrast', 'adapt_speed']].mean(axis=1)

print("Adaptation index statistics:")
print(adapt_df.groupby('subclass_short')[['adapt_contrast', 'adapt_speed']].describe().round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.2  Statistical tests and visualization of adaptation per cell type
# ══════════════════════════════════════════════════════════════════════

# ── Kruskal-Wallis across subclasses ──
for metric_name, metric_col in [('Contrast-context adapt.', 'adapt_contrast'),
                                 ('Speed-context adapt.', 'adapt_speed')]:
    groups = [adapt_df.loc[adapt_df['subclass'] == s, metric_col].dropna().values
              for s in present_subclasses]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        print(f"{metric_name}: Kruskal-Wallis H={stat:.2f}, p={p:.2e}")

# ── Paired Wilcoxon: is adaptation significantly != 0 per subclass? ──
print("\nOne-sample Wilcoxon (adaptation != 0):")
for sc in present_subclasses:
    for col_name, col in [('Contrast', 'adapt_contrast'), ('Speed', 'adapt_speed')]:
        vals = adapt_df.loc[adapt_df['subclass'] == sc, col].dropna().values
        if len(vals) >= 10:
            stat, p = wilcoxon(vals)
            median = np.median(vals)
            print(f"  {SUBCLASS_SHORT[sc]:10s} {col_name}: median={median:+.4f}, p={p:.2e}")

# ── Visualization: Violin plots of adaptation index ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

for ax, col, title in zip(axes, ['adapt_contrast', 'adapt_speed'],
                            ['Contrast-Context (Block 0 → 2)', 'Speed-Context (Block 1 → 3)']):
    sns.violinplot(data=adapt_df, x='subclass_short', y=col, order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.axhline(0, color='k', ls='--', alpha=0.5, lw=1)
    ax.set_title(f'C1: Adaptation Index — {title}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Adaptation Index\n(positive = weaker response in 2nd block)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# ── Scatter: Block 0 vs Block 2 mean response, colored by cell type ──
block0_mean = np.nanmean(np.column_stack([resp_block0[c] for c in contrast_conditions]), axis=1)
block2_mean = np.nanmean(np.column_stack([resp_block2[c] for c in contrast_conditions]), axis=1)

fig, axes = plt.subplots(1, len(present_subclasses), figsize=(4*len(present_subclasses), 4), sharex=True, sharey=True)
for ax, sc in zip(axes, present_subclasses):
    mask = obs['subclass_name'].values == sc
    ax.scatter(block0_mean[mask], block2_mean[mask], alpha=0.4, s=15, c=SUBCLASS_COLORS[sc])
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.4, lw=1)
    ax.set_title(SUBCLASS_SHORT[sc], color=SUBCLASS_COLORS[sc], fontweight='bold')
    ax.set_xlabel('Block 0 (1st contrast)')
    if ax == axes[0]:
        ax.set_ylabel('Block 2 (2nd contrast)')
plt.suptitle('C1: Paired Block Responses (Contrast Context)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.3  Within-block trial-position adaptation
# ══════════════════════════════════════════════════════════════════════

n_bins = 5
bin_labels = [f'Bin {i+1}' for i in range(n_bins)]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
block_pairs = [(0, 'Contrast Block 0'), (2, 'Contrast Block 2'),
               (1, 'Speed Block 1'), (3, 'Speed Block 3')]

for ax, (block, title) in zip(axes.flat, block_pairs):
    bin_resp = get_trial_position_responses(X, var, float(block), n_bins=n_bins)
    if bin_resp is None:
        continue
    for sc in present_subclasses:
        mask = obs['subclass_name'].values == sc
        if mask.sum() < 5:
            continue
        mean_trace = np.nanmean(bin_resp[mask], axis=0)
        sem_trace = np.nanstd(bin_resp[mask], axis=0) / np.sqrt(mask.sum())
        x = np.arange(n_bins)
        ax.errorbar(x, mean_trace, yerr=sem_trace, color=SUBCLASS_COLORS[sc],
                    label=SUBCLASS_SHORT[sc], linewidth=2, capsize=3, marker='o')
    ax.set_xticks(range(n_bins))
    ax.set_xticklabels(bin_labels)
    ax.set_xlabel('Trial Position (within block)')
    ax.set_ylabel('Mean ΔF/F')
    ax.set_title(f'C1: {title}', fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
plt.suptitle('C1: Within-Block Adaptation Time Course', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Fit exponential decay to within-block adaptation per subclass ──
print("\nExponential decay fit (within Block 0):")
bin_resp_b0 = get_trial_position_responses(X, var, 0.0, n_bins=10)
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    if mask.sum() < 10:
        continue
    y = np.nanmean(bin_resp_b0[mask], axis=0)
    x = np.arange(len(y))
    try:
        popt, _ = curve_fit(exp_decay, x, y, p0=[y[0]-y[-1], 3, y[-1]], maxfev=5000)
        print(f"  {SUBCLASS_SHORT[sc]:10s}: a={popt[0]:.4f}, tau={popt[1]:.2f} bins, baseline={popt[2]:.4f}")
    except:
        print(f"  {SUBCLASS_SHORT[sc]:10s}: fit failed")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C1.4  Adaptation heatmap: cells × conditions, sorted by type & magnitude
# ══════════════════════════════════════════════════════════════════════

# Build adaptation matrix per condition (contrast context)
adapt_matrix = np.zeros((adata.n_obs, len(contrast_conditions)))
for ci, cond in enumerate(contrast_conditions):
    r1 = resp_block0[cond]
    r2 = resp_block2[cond]
    denom = np.abs(r1) + np.abs(r2)
    denom[denom < 1e-8] = np.nan
    adapt_matrix[:, ci] = (r1 - r2) / denom

# Sort cells by subclass, then by mean adaptation
sort_df = pd.DataFrame({'subclass': obs['subclass_name'].values,
                         'mean_adapt': np.nanmean(adapt_matrix, axis=1)})
sort_df['subclass_rank'] = sort_df['subclass'].map(
    {s: i for i, s in enumerate(SUBCLASS_ORDER)}).fillna(99).astype(int)
sort_order = sort_df.sort_values(['subclass_rank', 'mean_adapt'], ascending=[True, False]).index

fig, ax = plt.subplots(figsize=(16, 10))
im = ax.imshow(adapt_matrix[sort_order], aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5,
               interpolation='none')
plt.colorbar(im, ax=ax, label='Adaptation Index')
ax.set_xlabel('Stimulus Condition (orientation × contrast)')
ax.set_ylabel('Cells (sorted by subclass)')

# Mark subclass boundaries
boundaries = []
for sc in present_subclasses:
    sc_mask = obs['subclass_name'].values[sort_order] == sc
    if sc_mask.any():
        boundaries.append((np.where(sc_mask)[0][0], SUBCLASS_SHORT[sc]))
for pos, label in boundaries:
    ax.axhline(pos, color='k', lw=0.5)
    ax.text(-2, pos + 10, label, fontsize=8, fontweight='bold', ha='right', va='center')

ax.set_title('C1: Adaptation Index per Cell × Condition (Contrast Context)', fontweight='bold')
plt.tight_layout()
plt.show()